# CarbinWatcher — Trash Detector Training (Hackathon Deployment Version)

Fine-tunes **YOLOv8n** (nano, smallest YOLOv8 variant) on a combined **TACO + Open Images v7 + Kaggle Garbage Classification (12 classes)** corpus (10,000+ images) and exports to **ONNX** for edge inference.

> **Reconstructed notebook.** The original notebook used for the hackathon's final on-device deployment was lost. This version rebuilds that pipeline from the surviving [train_classifier.ipynb](train_classifier.ipynb) (which trained YOLOv8x on ~1,500–8,500 images) with the two changes we know the deployed model used: the **nano** model variant and a **10,000+ image** training set. Exact dataset composition and hyperparameters from the original run are not recoverable — verify results before treating this as a drop-in replacement for the lost original.

> **Model choice — why nano, not x:** the edge device (`edge/linux/model_runner.py`) runs inference with `onnxruntime`'s `CPUExecutionProvider` — there is no GPU on the Arduino UNO Q, so `CUDAExecutionProvider` is never actually available at runtime (confirmed in the original notebook's own inference-verification output). YOLOv8n runs dramatically faster on CPU than YOLOv8x (~6 MB ONNX vs ~68 MB, far fewer FLOPs), which matters far more for a real-time detection loop than the accuracy YOLOv8x buys — hence the switch for the deployed build.

## Classes → bin mapping

| Category | Classes |
|----------|---------|
| recycle  | plastic_bottle, glass_bottle, metal_can, cardboard, paper, newspaper, aluminum_foil, beverage_carton |
| compost  | food_waste, fruit_peel, coffee_grounds, eggshell |
| landfill | styrofoam, plastic_bag, straw, tissue, chip_bag, dirty_container |
| hazardous | battery, electronics |

In [ ]:
# Install dependencies (Colab / fresh venv)
import subprocess, sys

packages = [
    'ultralytics>=8.0',
    'onnx>=1.14',
    'onnxruntime>=1.17',
    'requests',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'kagglehub',   # for the Kaggle Garbage Classification (12 classes) dataset
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import yaml
from ultralytics import YOLO

# ── Project paths ──────────────────────────────────────────────────────────
ROOT        = Path('carbinwatcher_data')
MODELS_DIR  = Path('../edge/linux/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset size target ────────────────────────────────────────────────────
# The final hackathon deployment trained on 10,000+ images. This is checked
# in section 1d below, once all three data sources have been merged.
TARGET_TOTAL_IMAGES = 10_000

# ── Class definitions ──────────────────────────────────────────────────────
CLASSES = [
    'plastic_bottle',    # 0  recycle
    'glass_bottle',      # 1  recycle
    'metal_can',         # 2  recycle
    'cardboard',         # 3  recycle
    'paper',             # 4  recycle
    'newspaper',         # 5  recycle
    'aluminum_foil',     # 6  recycle
    'beverage_carton',   # 7  recycle
    'food_waste',        # 8  compost
    'fruit_peel',        # 9  compost
    'coffee_grounds',    # 10 compost
    'eggshell',          # 11 compost
    'styrofoam',         # 12 landfill
    'plastic_bag',       # 13 landfill
    'straw',             # 14 landfill
    'tissue',            # 15 landfill
    'chip_bag',          # 16 landfill
    'dirty_container',   # 17 landfill
    'battery',           # 18 hazardous
    'electronics',       # 19 hazardous
]

LABEL_TO_CATEGORY = {
    'plastic_bottle': 'recycle',  'glass_bottle':   'recycle',
    'metal_can':      'recycle',  'cardboard':      'recycle',
    'paper':          'recycle',  'newspaper':      'recycle',
    'aluminum_foil':  'recycle',  'beverage_carton':'recycle',
    'food_waste':     'compost',  'fruit_peel':     'compost',
    'coffee_grounds': 'compost',  'eggshell':       'compost',
    'styrofoam':      'landfill', 'plastic_bag':    'landfill',
    'straw':          'landfill', 'tissue':         'landfill',
    'chip_bag':       'landfill', 'dirty_container':'landfill',
    'battery':        'hazardous','electronics':    'hazardous',
}

print(f'Tracking {len(CLASSES)} classes across 4 bin categories')

## 1. Dataset — TACO (Trash Annotations in Context)

**Dataset 1 of 3.** Open-source, no API key required.

The cell below:
1. Downloads the TACO annotation manifest (~15 MB) from GitHub.
2. Maps TACO's 60 COCO categories to our 20 canonical classes (unmapped categories are skipped).
3. Downloads up to `MAX_TRAIN_IMGS` / `MAX_VAL_IMGS` images from their source URLs.
4. Converts COCO `[x, y, w, h]` bounding boxes to YOLO normalised format.
5. Writes a `data.yaml` pointing at the prepared splits.

Lower the image limits for a quick smoke-test; set them to `None` to use the full corpus (~1,500 images — on its own, nowhere near the 10k target, which is why sections 1b/1c below add Open Images and Kaggle data on top).

In [ ]:
import json
import random
import requests
import yaml

# ── Download limits ────────────────────────────────────────────────────────
# None = use the full ~1 500-image TACO corpus.
MAX_TRAIN_IMGS = None
MAX_VAL_IMGS   = None

# ── TACO category name → canonical class ──────────────────────────────────
TACO_TO_CLASS = {
    'Plastic bottle':           'plastic_bottle',
    'Bottle':                   'plastic_bottle',
    'Glass bottle':             'glass_bottle',
    'Broken glass':             'glass_bottle',
    'Metal can':                'metal_can',
    'Drink can':                'metal_can',
    'Food Can':                 'metal_can',
    'Aluminium foil':           'aluminum_foil',
    'Drink carton':             'beverage_carton',
    'Juice carton':             'beverage_carton',
    'Milk carton':              'beverage_carton',
    'Corrugated carton':        'cardboard',
    'Egg carton':               'cardboard',
    'Paper':                    'paper',
    'Book':                     'paper',
    'Newspaper':                'newspaper',
    'Plastic bag + wrapper':    'plastic_bag',
    'Bag':                      'plastic_bag',
    'Garbage bag':              'plastic_bag',
    'Plastic film':             'plastic_bag',
    'Straw':                    'straw',
    'Paper straw':              'straw',
    'Tissues':                  'tissue',
    'Crisp packet':             'chip_bag',
    'Polystyrene item':         'styrofoam',
    'Styrofoam piece':          'styrofoam',
    'Foam food container':      'styrofoam',
    'Plastic container':        'dirty_container',
    'Dry food container':       'dirty_container',
    'Food container':           'dirty_container',
    'Spread tub':               'dirty_container',
    'Battery':                  'battery',
    'Blister pack':             'electronics',
}

CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

TACO_ANN_URL = (
    'https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json'
)

ROOT.mkdir(parents=True, exist_ok=True)
ann_path = ROOT / 'taco_annotations.json'

if not ann_path.exists():
    print('Fetching TACO annotations (~15 MB)…')
    r = requests.get(TACO_ANN_URL, timeout=120)
    r.raise_for_status()
    ann_path.write_bytes(r.content)
    print('Saved →', ann_path)
else:
    print('Annotations already cached →', ann_path)

with open(ann_path) as f:
    taco = json.load(f)

# ── Map TACO category IDs to canonical classes ─────────────────────────────
cat_id_to_class: dict[int, str] = {}
for cat in taco['categories']:
    mapped = TACO_TO_CLASS.get(cat['name'])
    if mapped:
        cat_id_to_class[cat['id']] = mapped

# ── Group annotations by image_id ─────────────────────────────────────────
img_anns: dict[int, list[dict]] = {}
for ann in taco['annotations']:
    if ann['category_id'] in cat_id_to_class:
        img_anns.setdefault(ann['image_id'], []).append(ann)

relevant_imgs = [img for img in taco['images'] if img['id'] in img_anns]
random.seed(42)
random.shuffle(relevant_imgs)

n_train = MAX_TRAIN_IMGS or int(len(relevant_imgs) * 0.8)
n_val   = MAX_VAL_IMGS   or (len(relevant_imgs) - n_train)
split_imgs = {
    'train': relevant_imgs[:n_train],
    'valid': relevant_imgs[n_train : n_train + n_val],
}
print(f'Relevant images — total: {len(relevant_imgs)}  '
      f'train: {len(split_imgs["train"])}  val: {len(split_imgs["valid"])}')
print(f'Mapped categories: {len(cat_id_to_class)} / {len(taco["categories"])}')


def coco_box_to_yolo(bbox: list, img_w: int, img_h: int) -> tuple:
    """COCO [x, y, w, h] → YOLO [cx, cy, bw, bh] normalised."""
    x, y, bw, bh = bbox
    return (x + bw / 2) / img_w, (y + bh / 2) / img_h, bw / img_w, bh / img_h


def download_split(split: str, images: list[dict]) -> None:
    img_dir = ROOT / split / 'images'
    lbl_dir = ROOT / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    ok = 0
    for img_info in images:
        img_id   = img_info['id']
        url      = img_info.get('flickr_url') or img_info.get('coco_url', '')
        stem     = f'taco_{img_id:06d}'
        img_path = img_dir / f'{stem}.jpg'
        lbl_path = lbl_dir / f'{stem}.txt'

        if not img_path.exists():
            try:
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                img_path.write_bytes(resp.content)
            except Exception as e:
                print(f'  skip {img_id}: {e}')
                continue

        w, h  = img_info['width'], img_info['height']
        lines = []
        for ann in img_anns.get(img_id, []):
            cls_name = cat_id_to_class[ann['category_id']]
            cls_idx  = CLASS_TO_IDX[cls_name]
            cx, cy, bw, bh = coco_box_to_yolo(ann['bbox'], w, h)
            lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

        lbl_path.write_text('\n'.join(lines) + '\n')
        ok += 1

    print(f'  {split}: {ok} images ready')


print('\nDownloading images (this may take a few minutes)…')
download_split('train', split_imgs['train'])
download_split('valid', split_imgs['valid'])

data_yaml = {
    'path':  str(ROOT.resolve()),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    len(CLASSES),
    'names': CLASSES,
}
DATASET_YAML = ROOT / 'data.yaml'
with open(DATASET_YAML, 'w') as f:
    yaml.dump(data_yaml, f)
print('\nDataset ready →', DATASET_YAML)

## 1b. Additional Data — Open Images v7 (via FiftyOne)

**Dataset 2 of 3.** Downloads up to `OI_MAX_TRAIN` / `OI_MAX_VAL` images (shared across all matched classes) from Google's Open Images v7 using the **FiftyOne** library (free, no API key required).
Only classes that map to our 20 canonical labels are fetched.
Bounding-box annotations are converted to YOLO format and written alongside the TACO images, so both datasets merge automatically into the same `data.yaml`.

Quotas are raised well above the original notebook's smoke-test values (500/100) since this is the largest lever for reaching the 10k+ image target — Open Images v7 has millions of annotated images per class, so supply isn't the constraint, only how much you choose to pull.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'fiftyone'])

import fiftyone as fo
import fiftyone.zoo as foz
import cv2
import shutil

# ── How many Open Images samples to pull per split ────────────────────────
# Shared across all matched classes below (FiftyOne's max_samples is a total,
# not a per-class quota) — raised from the 500/100 smoke-test values to
# actually move the needle towards the 10k+ target.
OI_MAX_TRAIN = 6000
OI_MAX_VAL   = 1200

# ── Open Images class label → canonical class ─────────────────────────────
# Only labels that have reliable bbox annotations in OI are listed.
OI_TO_CLASS: dict[str, str] = {
    'Bottle':              'plastic_bottle',
    'Plastic bottle':      'plastic_bottle',
    'Glass':               'glass_bottle',
    'Tin can':             'metal_can',
    'Aluminum can':        'metal_can',
    'Cardboard':           'cardboard',
    'Plastic bag':         'plastic_bag',
    'Paper bag':           'plastic_bag',
    'Drinking straw':      'straw',
    'Battery':             'battery',
    'Mobile phone':        'electronics',
    'Laptop':              'electronics',
    'Computer':            'electronics',
    'Tablet computer':     'electronics',
    'Tissue paper':        'tissue',
    'Paper':               'paper',
    'Newspaper':           'newspaper',
}

OI_CLASSES = list(OI_TO_CLASS.keys())


def _oi_split_to_yolo(fo_dataset: fo.Dataset, split_name: str) -> int:
    """Write Open Images detections into the existing YOLO split directories."""
    img_dir = ROOT / split_name / 'images'
    lbl_dir = ROOT / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    written = 0
    for sample in fo_dataset:
        src_path = sample.filepath
        if not src_path or not Path(src_path).exists():
            continue

        stem     = f'oi_{Path(src_path).stem}'
        dst_img  = img_dir / f'{stem}.jpg'
        dst_lbl  = lbl_dir / f'{stem}.txt'

        if not dst_img.exists():
            shutil.copy(src_path, dst_img)

        img = cv2.imread(str(dst_img))
        if img is None:
            continue
        h_img, w_img = img.shape[:2]

        lines = []
        detections = sample.ground_truth
        if detections is None:
            continue
        for det in detections.detections:
            oi_label = det.label
            canon    = OI_TO_CLASS.get(oi_label)
            if canon is None:
                continue
            cls_idx = CLASS_TO_IDX[canon]
            # FiftyOne bbox: [top-left-x, top-left-y, width, height] normalised
            x, y, bw, bh = det.bounding_box
            cx = x + bw / 2
            cy = y + bh / 2
            # Clamp to [0, 1]
            cx, cy, bw, bh = (min(max(v, 0.0), 1.0) for v in (cx, cy, bw, bh))
            if bw > 0 and bh > 0:
                lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

        if lines:
            dst_lbl.write_text('\n'.join(lines) + '\n')
            written += 1

    return written


for oi_split, yolo_split, max_n in [
    ('train',      'train', OI_MAX_TRAIN),
    ('validation', 'valid', OI_MAX_VAL),
]:
    print(f'\nFetching Open Images — {oi_split} (up to {max_n} images)…')
    try:
        oi_ds = foz.load_zoo_dataset(
            'open-images-v7',
            split=oi_split,
            label_types=['detections'],
            classes=OI_CLASSES,
            max_samples=max_n,
            only_matching=True,   # keep only samples that have ≥1 matching class
        )
        n = _oi_split_to_yolo(oi_ds, yolo_split)
        fo.delete_dataset(oi_ds.name)
        print(f'  Added {n} Open Images samples to {yolo_split}/')
    except Exception as e:
        print(f'  Open Images fetch failed ({e}) — continuing with TACO only.')

# Report running split sizes
for split in ('train', 'valid'):
    imgs = list((ROOT / split / 'images').glob('*.*'))
    print(f'{split}: {len(imgs)} total images after TACO + Open Images merge')

## 1c. Additional Data — Kaggle "Garbage Classification (12 classes)"

**Dataset 3 of 3.** Adds [mostafaabla/garbage-classification](https://www.kaggle.com/datasets/mostafaabla/garbage-classification) (~15,150 images) via `kagglehub`. This is the main lever for clearing 10,000 total images, since TACO + Open Images alone often fall short depending on how the shared OI quota distributes across classes.

**Caveats — read before trusting per-class metrics on these classes:**
- This is a **classification** dataset (one label per whole image, no bounding boxes), so each image is written as a single full-frame YOLO box (`cx=0.5, cy=0.5, bw=bh=0.9`). This is a common weak-supervision trick for bootstrapping a detector, but it teaches the model "this whole photo is class X," not tight localization — expect these classes to have better recall than precision on cluttered real-world frames.
- Its 12 classes don't map 1:1 onto our 20. Only classes with a reasonably faithful match are included below (`battery`, `cardboard`, `paper`, `biological → food_waste`, `metal → metal_can`, `*-glass → glass_bottle`). `clothes`, `shoes`, `trash`, and `plastic` are **skipped** — `plastic` in particular is too heterogeneous (bottles, bags, cutlery, wrap) to safely fold into `plastic_bottle` without adding label noise.
- **Verify the dataset's license/terms on the Kaggle page** before shipping a model trained on it — this wasn't confirmed at reconstruction time.
- Requires a Kaggle account: `kagglehub` will prompt for (or read from `KAGGLE_USERNAME`/`KAGGLE_KEY` env vars) Kaggle API credentials on first use.

In [ ]:
import kagglehub
import cv2

# ── Kaggle class folder name → canonical class ─────────────────────────────
# Deliberately excludes 'clothes', 'shoes', 'trash', 'plastic' — see markdown
# above for why. Folder names per the dataset's published structure.
KAGGLE_TO_CLASS: dict[str, str] = {
    'battery':      'battery',
    'cardboard':    'cardboard',
    'paper':        'paper',
    'biological':   'food_waste',
    'metal':        'metal_can',
    'brown-glass':  'glass_bottle',
    'green-glass':  'glass_bottle',
    'white-glass':  'glass_bottle',
}

# Kaggle dataset ships as one pool of images per class (no pre-split
# train/val) — we split each class 85/15 ourselves to match TACO/OI's split.
KAGGLE_VAL_FRACTION = 0.15


def _write_full_frame_label(lbl_path: Path, cls_idx: int, box_frac: float = 0.9) -> None:
    pad = (1 - box_frac) / 2
    lbl_path.write_text(f'{cls_idx} 0.500000 0.500000 {box_frac:.6f} {box_frac:.6f}\n')


try:
    kaggle_root = Path(kagglehub.dataset_download('mostafaabla/garbage-classification'))
    print('Kaggle dataset cached at', kaggle_root)

    added = {'train': 0, 'valid': 0}
    random.seed(43)

    for folder_name, canon in KAGGLE_TO_CLASS.items():
        cls_idx = CLASS_TO_IDX[canon]
        class_dirs = list(kaggle_root.rglob(folder_name))
        images = [p for d in class_dirs for p in d.glob('*.*')
                  if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        if not images:
            print(f'  no images found for folder "{folder_name}" — skipping')
            continue

        random.shuffle(images)
        n_val = max(1, int(len(images) * KAGGLE_VAL_FRACTION))
        split_files = {'valid': images[:n_val], 'train': images[n_val:]}

        for split_name, files in split_files.items():
            img_dir = ROOT / split_name / 'images'
            lbl_dir = ROOT / split_name / 'labels'
            img_dir.mkdir(parents=True, exist_ok=True)
            lbl_dir.mkdir(parents=True, exist_ok=True)

            for src_img in files:
                stem    = f'kg_{folder_name}_{src_img.stem}'
                dst_img = img_dir / f'{stem}.jpg'
                dst_lbl = lbl_dir / f'{stem}.txt'
                if not dst_img.exists():
                    img = cv2.imread(str(src_img))
                    if img is None:
                        continue
                    cv2.imwrite(str(dst_img), img)
                _write_full_frame_label(dst_lbl, cls_idx)
                added[split_name] += 1

        print(f'  {folder_name} → {canon}: {len(images)} images '
              f'({len(split_files["train"])} train / {len(split_files["valid"])} val)')

    print(f'\nAdded {added["train"]} train / {added["valid"]} val images from Kaggle')
except Exception as e:
    print(f'Kaggle dataset fetch failed ({e}) — continuing without it.')
    print('If this is a credentials error, set KAGGLE_USERNAME / KAGGLE_KEY and re-run this cell.')

## 1d. Dataset size check

Confirms the merged TACO + Open Images + Kaggle corpus actually clears the 10,000-image target before spending compute on training.

In [ ]:
total_images = 0
for split in ('train', 'valid'):
    n = len(list((ROOT / split / 'images').glob('*.*')))
    print(f'{split}: {n} images')
    total_images += n

print(f'\nTotal: {total_images} images (target: {TARGET_TOTAL_IMAGES})')
if total_images < TARGET_TOTAL_IMAGES:
    print(f'⚠️  {TARGET_TOTAL_IMAGES - total_images} images short of target — '
          f'raise OI_MAX_TRAIN/OI_MAX_VAL in 1b, or check the Kaggle fetch in 1c succeeded.')
else:
    print('Target reached.')

## 2. Verify data.yaml

The download cell already writes `data.yaml` with the canonical class list.
This cell confirms the config looks correct before training.

In [ ]:
with open(DATASET_YAML) as f:
    data_cfg = yaml.safe_load(f)

print('Original classes:', data_cfg.get('names', []))

# If nc matches, assume names are already aligned.
# Otherwise update names (and nc) to our canonical list — verify alignment manually.
if data_cfg.get('nc', 0) != len(CLASSES):
    print(f'⚠️  nc mismatch ({data_cfg["nc"]} vs {len(CLASSES)}) — updating names to canonical list.')
    print('   Verify that label indices in your dataset match CLASSES order above!')
    data_cfg['nc']    = len(CLASSES)
    data_cfg['names'] = CLASSES
    with open(DATASET_YAML, 'w') as f:
        yaml.dump(data_cfg, f)

print('\nFinal classes:', data_cfg['names'])

## 3. Fine-tune YOLOv8n

YOLOv8n is the **nano** variant (~3.2 M parameters, ~6 MB ONNX).
Training on Colab T4 (15 GB VRAM) — nano's small memory footprint means `BATCH_SIZE` can go much higher than the x-variant's batch-8 ceiling.
Inference on the Arduino UNO Q runs on CPU only (`onnxruntime` `CPUExecutionProvider` — there is no GPU on-device), which is the actual reason this variant was chosen for deployment: nano's far lower compute cost matters more for a real-time CPU detection loop than the accuracy headroom YOLOv8x offers.

In [ ]:
# Hyper-parameters
EPOCHS      = 50
IMG_SIZE    = 640
BATCH_SIZE  = 32     # nano needs far less VRAM than v8x — 32 is safe on a T4
LR0         = 0.01
PATIENCE    = 10     # early-stopping patience (epochs without improvement)
PROJECT_DIR = 'runs/carbinwatcher'
RUN_NAME    = 'yolov8n_trash'

model = YOLO('yolov8n.pt')  # download pretrained nano weights

results = model.train(
    data      = str(DATASET_YAML),
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    lr0       = LR0,
    patience  = PATIENCE,
    project   = PROJECT_DIR,
    name      = RUN_NAME,
    # Augmentation
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    copy_paste= 0.1,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.5,
    shear     = 2.0,
    # Device
    device    = 0 if __import__('torch').cuda.is_available() else 'cpu',
    verbose   = True,
)

BEST_WEIGHTS = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'
print('\nBest weights saved →', BEST_WEIGHTS)

## 4. Evaluate

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics    = best_model.val(data=str(DATASET_YAML), imgsz=IMG_SIZE, verbose=True)

print(f'\nmAP@50:     {metrics.box.map50:.4f}')
print(f'mAP@50-95:  {metrics.box.map:.4f}')
print(f'Precision:  {metrics.box.mp:.4f}')
print(f'Recall:     {metrics.box.mr:.4f}')

In [ ]:
# Per-class AP bar chart
# NOTE: metrics.box.ap50 only contains entries for classes that had at least
# one ground-truth instance in the validation split — it is NOT guaranteed to
# be length len(CLASSES). Aligning positionally (as the original x-variant
# notebook did) crashes with a broadcast ValueError whenever any class has
# zero val instances, which is common with a mixed TACO/OI/Kaggle corpus.
# Fix: scatter onto a full-length array using ap_class_index, leaving classes
# with no validation support at 0.
ap_per_class = np.zeros(len(CLASSES))
ap_per_class[metrics.box.ap_class_index] = metrics.box.ap50

missing = [CLASSES[i] for i in range(len(CLASSES)) if i not in metrics.box.ap_class_index]
if missing:
    print(f'⚠️  No validation instances for: {missing} — plotted as 0 AP@50, not "no detections"')

cat_colors = {
    'recycle':   '#4CAF50',
    'compost':   '#8D6E63',
    'landfill':  '#9E9E9E',
    'hazardous': '#F44336',
}
colors = [cat_colors[LABEL_TO_CATEGORY[c]] for c in CLASSES]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(CLASSES, ap_per_class, color=colors)
ax.set_ylabel('AP@50')
ax.set_title('Per-class AP@50')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.axhline(float(np.mean(ap_per_class)), color='navy', linestyle='--', label='mean')
ax.legend()
plt.tight_layout()
plt.savefig('ap_per_class.png', dpi=150)
plt.show()

## 5. Export to ONNX

In [ ]:
onnx_path = best_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    dynamic  = False,   # fixed input shape for deterministic edge inference
    simplify = True,    # onnx-simplifier reduces graph complexity
    opset    = 17,
)
print('ONNX model exported →', onnx_path)

In [ ]:
import shutil

# Copy ONNX model and labels to edge/linux/models/
dest_model  = MODELS_DIR / 'trash_detector.onnx'
dest_labels = MODELS_DIR / 'labels.txt'

shutil.copy(onnx_path, dest_model)
dest_labels.write_text('\n'.join(CLASSES) + '\n')

print('Copied to', dest_model)
print('Labels  →', dest_labels)

## 6. Verify ONNX inference with onnxruntime

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(
    str(dest_model),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)

input_name  = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name
print('Input  :', input_name, sess.get_inputs()[0].shape)
print('Output :', output_name, sess.get_outputs()[0].shape)

# Dummy inference with a random frame
dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
out   = sess.run([output_name], {input_name: dummy})[0]
print('Output shape:', out.shape)  # expect [1, 24, 8400] — 4 box coords + 20 classes

# Quick sanity: highest confidence across all anchors
pred         = out[0].T                            # [8400, 24]
scores       = pred[:, 4:].max(axis=1)
top_idx      = scores.argmax()
top_class    = pred[top_idx, 4:].argmax()
print(f'Highest confidence: {scores[top_idx]:.4f} → class {top_class} ({CLASSES[top_class]})')

## 7. Visualise predictions on a sample image

In [ ]:
import glob

# Grab the first validation image
val_images = glob.glob(str(ROOT / 'valid' / 'images' / '*.*'))
if not val_images:
    val_images = glob.glob(str(ROOT / 'train' / 'images' / '*.*'))

sample_path = val_images[0]
frame       = cv2.imread(sample_path)
h_orig, w_orig = frame.shape[:2]

# Preprocess — letterbox
s     = IMG_SIZE
scale = min(s / w_orig, s / h_orig)
nw, nh = int(w_orig * scale), int(h_orig * scale)
pad_x, pad_y = (s - nw) // 2, (s - nh) // 2
canvas = np.full((s, s, 3), 114, dtype=np.uint8)
canvas[pad_y:pad_y+nh, pad_x:pad_x+nw] = cv2.resize(frame, (nw, nh))
blob   = (canvas[:, :, ::-1].astype(np.float32) / 255.0)
blob   = np.transpose(blob, (2, 0, 1))[np.newaxis]

# Inference
raw   = sess.run([output_name], {input_name: blob})[0][0].T  # [8400, 84]
confs = raw[:, 4:].max(axis=1)
clsids = raw[:, 4:].argmax(axis=1)
mask  = confs >= 0.45

CATEGORY_COLOR = {
    'recycle':   (76, 175, 80),
    'compost':   (141, 110, 99),
    'landfill':  (158, 158, 158),
    'hazardous': (244, 67, 54),
}

vis = frame.copy()
for i in np.where(mask)[0]:
    cx, cy, bw, bh = raw[i, :4]
    x1 = int(((cx - bw/2) - pad_x) / scale)
    y1 = int(((cy - bh/2) - pad_y) / scale)
    x2 = int(((cx + bw/2) - pad_x) / scale)
    y2 = int(((cy + bh/2) - pad_y) / scale)
    label    = CLASSES[clsids[i]]
    category = LABEL_TO_CATEGORY[label]
    color    = CATEGORY_COLOR[category]
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    cv2.putText(vis, f'{label} {confs[i]:.2f}', (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

plt.figure(figsize=(10, 7))
plt.imshow(vis[:, :, ::-1])
plt.axis('off')
plt.title(f'Sample detections — {Path(sample_path).name}')
plt.tight_layout()
plt.show()

## 8. Deployment checklist

| Step | Command |
|------|---------|
| Copy model to device | `scp edge/linux/models/trash_detector.onnx user@arduinounoq:~/carbinwatcher/models/` |
| Flash MCU firmware | `cd edge/mcu && pio run -t upload` |
| Set env vars | `export BIN_LEFT=recycle BIN_RIGHT=landfill GEMINI_API_KEY=... S3_BUCKET=...` |
| Run detector | `cd edge/linux && python detect.py` |
| Monitor serial | `cd edge/mcu && pio device monitor` |

**Calibrate focal length** once per camera:  
Hold a 100 mm-wide object at a known distance (e.g. 300 mm).  
Measure `object_width_px` in the frame, then:  
`FOCAL_LENGTH_PX = object_width_px * 300 / 100`  
Set via `export FOCAL_LENGTH_PX=<value>`.